In [52]:
from transformers import BertTokenizerFast, AutoModelForCausalLM
import torch
import numpy as np

In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [18]:
tokenizer = BertTokenizerFast.from_pretrained("bert-base-chinese")
model = AutoModelForCausalLM.from_pretrained("ckiplab/gpt2-base-chinese")

Loading weights: 100%|██████████| 149/149 [00:00<00:00, 50954.04it/s]
[transformers] GPT2LMHeadModel LOAD REPORT from: ckiplab/gpt2-base-chinese
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [130]:
SPECIAL_TOKENS = "[CLS]", "[SEP]", "[PAD]"
SPECIAL_TOKEN_IDS = 0, 101, 102  # BERT token IDs: 0 = [PAD]; 101 = [CLS]; 102 = [SEP]

def get_word_logprobs(*, tokens: list, split_sentences: list,
                      token_logprobs: np.ndarray,
                      special_tokens: tuple | list | None=None,
                      special_token_ids: tuple| list| None=None) -> list:
    if special_tokens is None or special_token_ids is None:   
        special_tokens = SPECIAL_TOKENS
        special_token_ids = SPECIAL_TOKEN_IDS

    tokens = [list(filter(lambda s: s not in special_tokens, t)) for t in tokens]
    
    spans = [[0] for _ in tokens]
    for i, (trow, srow) in enumerate(zip(tokens, split_sentences)):
        chunk = ""
        idx = 0
        for j, t in enumerate(trow):
            if chunk in srow[idx] and chunk != srow[idx]:
                chunk += t
            else:
                chunk = t
                idx += 1
                spans[i].append(j)
        spans[i].append(len(tokens[i]))
    
    token_logprobs_masked = [tprobs[~ np.isin(tprobs, special_token_ids)] for tprobs in token_logprobs]
    word_logprobs = [[sum(tprobs[seq[i]: seq[i+1]]) for i in range(len(seq)-1)]
                                                    for tprobs, seq in zip(token_logprobs_masked, spans)]
    return word_logprobs


sentences = ["這對夫妻攜手走過四十年現在依然非常",
        "他們依偎彼此手牽手漫步在沙灘上看來非常"]
split_sentences = [["這對", "夫妻", "攜手", "走過", "四十年", "現在","依然","非常"],
        ["他們", "依偎", "彼此", "手牽手", "漫步在", "沙灘上", "看來" ,"非常"]]

inputs = tokenizer(sentences,
                   truncation=True,
                   padding=True,
                   return_tensors="pt",
                   return_offsets_mapping=False,
                   max_length=256)
inputs = {k: v.to(device) for k, v in inputs.items()}
ids = inputs["input_ids"]      
tokens = [tokenizer.convert_ids_to_tokens(i) for i in ids]

# offsets = inputs["offset_mapping"]
# inputs.pop("offset_mapping") # <- don't need offset to map tokens back to "words"

with torch.inference_mode():
    outputs = model(**inputs,
                    output_hidden_states=False)
logits = outputs.logits    # shape = (batch_size, seq_len, vocab_size)
logprobs = torch.log_softmax(logits, dim=-1)
logprobs = logprobs.detach().cpu().numpy()

indices = ids[:, 1:]
token_logprobs = np.array([[logprobs[i, j, idx] for j, idx in enumerate(indices[i])]
                                       for i in range(indices.size(0))])
token_logprobs = np.concatenate((np.zeros((indices.size(0), 1)),
                                 token_logprobs),
                                 axis=1)

special_token_ids = SPECIAL_TOKEN_IDS  
word_logprobs = get_word_logprobs(tokens=tokens,
                                  split_sentences=split_sentences,
                                  token_logprobs=token_logprobs,
                                  special_tokens=SPECIAL_TOKENS,
                                  special_token_ids=SPECIAL_TOKEN_IDS)
print(word_logprobs)


[[np.float64(-10.790741920471191), np.float64(-6.3987040519714355), np.float64(-8.881091475486755), np.float64(-7.107176303863525), np.float64(-8.212314307689667), np.float64(-9.81073135137558), np.float64(-7.747424393892288), np.float64(-5.378808066248894)], [np.float64(-7.981077671051025), np.float64(-13.732409954071045), np.float64(-9.1331507563591), np.float64(-16.962288856506348), np.float64(-18.265514850616455), np.float64(-7.46296352148056), np.float64(-10.648432731628418), np.float64(-5.095374420285225)]]
